# Automotive Supply Chain Multi-Agent Control Tower (v2 — adaptable / SCM-connector architecture)

This version is built so the **agent and orchestration logic is decoupled from the data source**, using a connector interface — the same pattern SAP, Oracle, and Zoho integrations are built on. Today the connector reads synthetic data generated in this notebook. Swapping in a real system means writing a new connector class against that system's API; **the agents and orchestrator below never change.**

**What's new vs. v1:**
- `SCMConnector` abstract interface + notes on what a real `SAPConnector` / `OracleConnector` / `ZohoConnector` would implement it against
- Agents are now dynamic — they query data and score real candidates, not fixed hardcoded numbers
- Richer synthetic dataset: multiple suppliers per part per tier, 12 weeks of demand history, plant capacity, a carrier/logistics options table
- Configurable policy weights (mirrors how SAP IBP / Oracle SCM Cloud expose tunable planning parameters, not hardcoded constants)
- Two disruption scenarios run through the same pipeline, to demonstrate the logic generalizes rather than being scenario-specific

**Still true (say this plainly in any writeup):** all data below is synthetic. No live SAP/Oracle/Zoho connection is made in this notebook — there's no network access in this environment, and doing so requires real credentials this notebook doesn't have. What's proven here is the *architecture pattern* for that integration, not the integration itself.

> **Run this notebook top to bottom without skipping cells** (Runtime -> Run all, or Runtime -> Restart session and run all). Every cell below depends on variables defined in an earlier cell -- if you re-run a single cell on its own after a restart, or run cells out of order, you will get a `NameError` for a variable that looks like it should exist. That error means 'an earlier cell hasn't run in this session yet,' not a bug in the code.

## 1. Setup

In [2]:
!pip install -q networkx pandas numpy matplotlib

import random
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
from IPython.display import display, HTML

random.seed(7)
np.random.seed(7)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)


def render_agent_report(title, subtitle, proposals, decision):
    """Renders a clean HTML report card: what each agent recommends, its cost/time/
    confidence, and the combined final plan -- instead of a raw text trace dump."""
    rows_html = ''
    for p in proposals:
        cost_color = '#c0392b' if p.cost_delta_pct > 0 else '#27ae60'
        time_color = '#c0392b' if p.delay_days > 0 else '#27ae60'
        rows_html += f'''
        <tr>
          <td style="padding:8px 12px;font-weight:600;border-bottom:1px solid #eee;">{p.agent}</td>
          <td style="padding:8px 12px;border-bottom:1px solid #eee;">{p.option}</td>
          <td style="padding:8px 12px;border-bottom:1px solid #eee;color:{cost_color};">{p.cost_delta_pct:+.1f}%</td>
          <td style="padding:8px 12px;border-bottom:1px solid #eee;color:{time_color};">{p.delay_days:+.1f} days</td>
          <td style="padding:8px 12px;border-bottom:1px solid #eee;">{p.confidence:.0%}</td>
          <td style="padding:8px 12px;border-bottom:1px solid #eee;color:#555;font-size:0.9em;">{p.risk_note}</td>
        </tr>'''
    plan_color = '#c0392b' if (decision['total_cost_pct'] > 0 or decision['total_delay_days'] > 0) else '#27ae60'
    html = f'''
    <div style="border:1px solid #ddd;border-radius:10px;padding:16px 20px;margin:12px 0;
                font-family:-apple-system,Segoe UI,Roboto,sans-serif;background:#fafafa;">
      <h3 style="margin:0 0 4px 0;color:#222;">{title}</h3>
      <p style="margin:0 0 12px 0;color:#666;">{subtitle}</p>
      <table style="width:100%;border-collapse:collapse;background:white;border-radius:6px;overflow:hidden;">
        <thead>
          <tr style="background:#2c3e50;color:white;text-align:left;">
            <th style="padding:8px 12px;">Agent</th>
            <th style="padding:8px 12px;">Recommendation</th>
            <th style="padding:8px 12px;">Cost Impact</th>
            <th style="padding:8px 12px;">Time Impact</th>
            <th style="padding:8px 12px;">Confidence</th>
            <th style="padding:8px 12px;">Why</th>
          </tr>
        </thead>
        <tbody>{rows_html}</tbody>
      </table>
      <div style="margin-top:12px;padding:10px 14px;background:#eef3f8;border-left:4px solid {plan_color};border-radius:4px;">
        <b>Final plan:</b> cost {decision['total_cost_pct']:+.1f}% &nbsp;|&nbsp;
        schedule {decision['total_delay_days']:+.1f} days &nbsp;|&nbsp;
        confidence {decision['avg_confidence']:.0%}
        <br><span style="color:#555;font-size:0.9em;">Proceed with the combined plan above, pending manager approval.</span>
      </div>
    </div>
    '''
    display(HTML(html))


def _highlight_status(val):
    """Row color coding for a 'status' column: red for AT RISK, green for OK."""
    if val == 'AT RISK':
        return 'background-color:#fdecea;color:#c0392b;font-weight:600;'
    return 'background-color:#eafaf1;color:#1e8449;font-weight:600;'


def style_scan_results(df):
    """Applies consistent, readable formatting to any scan/monitoring summary table
    with 'status', 'buffer_days', 'cost_impact_pct', 'schedule_impact_days' columns."""
    styler = df.style.format({
        'buffer_days': '{:.1f} days',
        'cost_impact_pct': '{:+.1f}%',
        'schedule_impact_days': '{:+.1f} days',
    })
    try:
        styler = styler.map(_highlight_status, subset=['status'])
    except AttributeError:
        # older pandas versions use applymap instead of map on a Styler
        styler = styler.applymap(_highlight_status, subset=['status'])
    return styler



def generate_synthetic_data():
    """Builds the full synthetic dataset and returns it as a dict of
    variable_name -> value, ready to be dropped into globals()."""
    plants = [
        {'plant_id': 'PLANT_A', 'name': 'Riverside Assembly', 'region': 'North America', 'weekly_capacity_units': 4200},
        {'plant_id': 'PLANT_B', 'name': 'Meridian Works', 'region': 'Europe', 'weekly_capacity_units': 3100},
    ]

    models = [
        {'model_id': 'MODEL_SUV1', 'name': 'Voyager SUV', 'plant_id': 'PLANT_A'},
        {'model_id': 'MODEL_SED1', 'name': 'Aria Sedan', 'plant_id': 'PLANT_A'},
        {'model_id': 'MODEL_SUV2', 'name': 'Terra Crossover', 'plant_id': 'PLANT_B'},
    ]

    parts = [
        {'part_id': 'PART_ECU',  'name': 'Infotainment ECU chip', 'category': 'electronics'},
        {'part_id': 'PART_SEAT', 'name': 'Seat frame assembly',   'category': 'interior'},
        {'part_id': 'PART_BATT', 'name': 'EV battery module',     'category': 'powertrain'},
        {'part_id': 'PART_TIRE', 'name': 'Tire set',               'category': 'chassis'},
    ]

    bom = {
        'MODEL_SUV1': ['PART_ECU', 'PART_SEAT', 'PART_TIRE'],
        'MODEL_SED1': ['PART_ECU', 'PART_SEAT', 'PART_TIRE'],
        'MODEL_SUV2': ['PART_ECU', 'PART_BATT', 'PART_TIRE'],
    }

    # --- Multiple suppliers per part per tier, with realistic operational fields ---
    supplier_rows = [
        # part, tier, name, region, reliability(0-1), financial_health(0-1), lead_time_days, unit_cost_usd, weekly_capacity, certified
        ('PART_ECU', 1, 'Nexlight Electronics',   'Taiwan',      0.91, 0.88, 12, 42.0, 5000, True),
        ('PART_ECU', 2, 'Coreway Semiconductors', 'South Korea', 0.85, 0.80, 20, 18.5, 4200, True),
        ('PART_ECU', 2, 'Brightline Semi',        'Vietnam',     0.78, 0.74, 26, 21.8, 3000, True),
        ('PART_ECU', 3, 'Silvex Wafer Fab',       'Japan',       0.93, 0.90, 35, 6.2,  6000, True),
        ('PART_ECU', 3, 'Quarzon Materials',      'Germany',     0.82, 0.86, 40, 7.1,  2800, True),
        ('PART_SEAT', 1, 'Comfort Interiors Ltd', 'Mexico',      0.88, 0.83, 9,  61.0, 6000, True),
        ('PART_SEAT', 1, 'Padwell Systems',       'Poland',      0.80, 0.77, 11, 58.5, 4500, True),
        ('PART_BATT', 1, 'VoltCore Batteries',    'Germany',     0.89, 0.91, 18, 310.0, 2500, True),
        ('PART_BATT', 1, 'Ionix Power',           'Sweden',      0.83, 0.85, 22, 295.0, 1800, True),
        ('PART_TIRE', 1, 'GripLine Tires',        'Thailand',    0.90, 0.87, 7,  38.0, 8000, True),
    ]
    suppliers = pd.DataFrame(supplier_rows, columns=[
        'part_id', 'tier', 'name', 'region', 'reliability', 'financial_health',
        'lead_time_days', 'unit_cost_usd', 'weekly_capacity', 'certified'
    ])
    suppliers['supplier_id'] = ['SUP_' + str(i).zfill(3) for i in range(len(suppliers))]

    # Upstream feed relationships (which Tier2/3 suppliers feed which Tier1, for graph tracing)
    supplier_feeds = [
        ('SUP_000', 'SUP_001'),  # Nexlight (T1) fed by Coreway (T2)
        ('SUP_001', 'SUP_003'),  # Coreway (T2) fed by Silvex (T3)
        ('SUP_000', 'SUP_002'),  # Nexlight (T1) also fed by Brightline (T2)
        ('SUP_002', 'SUP_004'),  # Brightline (T2) fed by Quarzon (T3)
    ]

    # --- 12 weeks of demand history per model (units/week), with mild seasonality + noise ---
    weeks = list(range(1, 13))
    base_demand = {'MODEL_SUV1': 900, 'MODEL_SED1': 650, 'MODEL_SUV2': 700}
    demand_history = []
    for model_id, base in base_demand.items():
        for w in weeks:
            seasonal = 1 + 0.05 * np.sin(w / 2)
            noise = np.random.normal(0, 0.04)
            demand_history.append({'model_id': model_id, 'week': w, 'units': max(0, round(base * (seasonal + noise)))})
    demand_history = pd.DataFrame(demand_history)

    # --- Current inventory snapshot per part/plant, in on-hand units ---
    inventory = pd.DataFrame([
        {'part_id': 'PART_ECU',  'plant_id': 'PLANT_A', 'on_hand_units': 5400},
        {'part_id': 'PART_ECU',  'plant_id': 'PLANT_B', 'on_hand_units': 2600},
        {'part_id': 'PART_SEAT', 'plant_id': 'PLANT_A', 'on_hand_units': 9000},
        {'part_id': 'PART_BATT', 'plant_id': 'PLANT_B', 'on_hand_units': 1900},
        {'part_id': 'PART_TIRE', 'plant_id': 'PLANT_A', 'on_hand_units': 12000},
    ])

    # --- Logistics / carrier options table ---
    carriers = pd.DataFrame([
        {'mode': 'sea',  'days_transit': 24, 'cost_index': 1.00},
        {'mode': 'rail', 'days_transit': 14, 'cost_index': 1.35},
        {'mode': 'air',  'days_transit': 5,  'cost_index': 2.60},
    ])

    return {
        'plants': plants, 'models': models, 'parts': parts, 'bom': bom,
        'suppliers': suppliers, 'supplier_feeds': supplier_feeds,
        'demand_history': demand_history, 'inventory': inventory, 'carriers': carriers,
    }


print('Environment ready.')


Environment ready.


## 2. Synthetic data generator



In [3]:
_synthetic_data = generate_synthetic_data()
globals().update(_synthetic_data)

print(f'{len(suppliers)} supplier records, {len(demand_history)} demand rows, {len(carriers)} carrier modes loaded.')
suppliers.head(10)


10 supplier records, 36 demand rows, 3 carrier modes loaded.


,part_id,tier,name,region,reliability,financial_health,lead_time_days,unit_cost_usd,weekly_capacity,certified,supplier_id
0,PART_ECU,1,Nexlight Electronics,Taiwan,0.91,0.88,12,42.0,5000,True,SUP_000
1,PART_ECU,2,Coreway Semiconductors,South Korea,0.85,0.80,20,18.5,4200,True,SUP_001
2,PART_ECU,2,Brightline Semi,Vietnam,0.78,0.74,26,21.8,3000,True,SUP_002
3,PART_ECU,3,Silvex Wafer Fab,Japan,0.93,0.90,35,6.2,6000,True,SUP_003
4,PART_ECU,3,Quarzon Materials,Germany,0.82,0.86,40,7.1,2800,True,SUP_004
5,PART_SEAT,1,Comfort Interiors Ltd,Mexico,0.88,0.83,9,61.0,6000,True,SUP_005
6,PART_SEAT,1,Padwell Systems,Poland,0.80,0.77,11,58.5,4500,True,SUP_006
7,PART_BATT,1,VoltCore Batteries,Germany,0.89,0.91,18,310.0,2500,True,SUP_007
8,PART_BATT,1,Ionix Power,Sweden,0.83,0.85,22,295.0,1800,True,SUP_008
9,PART_TIRE,1,GripLine Tires,Thailand,0.90,0.87,7,38.0,8000,True,SUP_009


## 3. The connector abstraction — this is what makes it adaptable to SAP / Oracle / Zoho

`SCMConnector` defines the fixed contract every agent depends on. Today `SyntheticSCMConnector` implements it against the synthetic tables above. To point this system at a real SCM platform, you write a new class implementing the same methods against that platform's real API — the agents in Section 4 call the connector interface, never the underlying tables directly, so **nothing downstream changes**.


In [4]:

_required = ['suppliers', 'supplier_feeds', 'inventory', 'demand_history', 'plants', 'carriers']
_missing = [v for v in _required if v not in dir()]
if _missing:
    if 'generate_synthetic_data' in dir():
        print(f'Note: {_missing} were not found in this session -- regenerating synthetic data automatically.')
        globals().update(generate_synthetic_data())
    else:
        raise RuntimeError(
            'Section 1 (Setup) has not run in this session, so the tools needed to even regenerate data '
            '(pandas, numpy, generate_synthetic_data) are unavailable. '
            'Go to Runtime -> Restart session and run all -- this is the one thing that must run first.'
        )

class SCMConnector(ABC):
    """Fixed interface every agent and the orchestrator depend on.
    Swap the implementation to change data source; agent logic never changes."""

    @abstractmethod
    def get_suppliers(self, part_id: str) -> pd.DataFrame: ...

    @abstractmethod
    def get_supplier_upstream(self, supplier_id: str) -> List[str]: ...

    @abstractmethod
    def get_inventory(self, part_id: str, plant_id: str) -> int: ...

    @abstractmethod
    def get_avg_weekly_demand(self, model_id: str) -> float: ...

    @abstractmethod
    def get_plant_capacity(self, plant_id: str) -> int: ...

    @abstractmethod
    def get_logistics_options(self) -> pd.DataFrame: ...

    @abstractmethod
    def post_decision(self, decision: dict) -> None:
        """Write an approved decision back to the source system
        (e.g. create a PO, update a production schedule, book a shipment)."""
        ...


class SyntheticSCMConnector(SCMConnector):
    def __init__(self, suppliers, supplier_feeds, inventory, demand_history, plants, carriers):
        self.suppliers = suppliers
        self.supplier_feeds = supplier_feeds
        self.inventory = inventory
        self.demand_history = demand_history
        self.plants = {p['plant_id']: p for p in plants}
        self.carriers = carriers

    def get_suppliers(self, part_id):
        return self.suppliers[self.suppliers['part_id'] == part_id].copy()

    def get_supplier_upstream(self, supplier_id):
        return [downstream for downstream, upstream in self.supplier_feeds if upstream == supplier_id] + \
               [upstream for downstream, upstream in self.supplier_feeds if downstream == supplier_id]

    def get_inventory(self, part_id, plant_id):
        row = self.inventory[(self.inventory['part_id'] == part_id) & (self.inventory['plant_id'] == plant_id)]
        return int(row['on_hand_units'].iloc[0]) if len(row) else 0

    def get_avg_weekly_demand(self, model_id):
        return float(self.demand_history[self.demand_history['model_id'] == model_id]['units'].mean())

    def get_plant_capacity(self, plant_id):
        return self.plants[plant_id]['weekly_capacity_units']

    def get_logistics_options(self):
        return self.carriers.copy()

    def post_decision(self, decision: dict) -> None:
        print(f"[connector] would write back to source system: {decision['action_type']} -> {decision['summary']}")



connector = SyntheticSCMConnector(suppliers, supplier_feeds, inventory, demand_history, plants, carriers)
print('Connector ready:', connector.__class__.__name__)


Note: ['suppliers', 'supplier_feeds', 'inventory', 'demand_history', 'plants', 'carriers'] were not found in this session -- regenerating synthetic data automatically.
Connector ready: SyntheticSCMConnector


## 4. Knowledge graph (built from the connector, not raw tables)

Same purpose as v1 — trace a disruption through tiers to the plants it affects — but now sourced through the connector so it stays correct if the underlying data source changes.

In [5]:
def build_graph(connector, plants, models, bom, parts):
    G = nx.DiGraph()
    for p in plants:
        G.add_node(p['plant_id'], type='plant', label=p['name'])
    for m in models:
        G.add_node(m['model_id'], type='model', label=m['name'])
        G.add_edge(m['model_id'], m['plant_id'], relation='assembled_at')
    for part in parts:
        G.add_node(part['part_id'], type='part', label=part['name'])
    for model_id, part_list in bom.items():
        for part_id in part_list:
            G.add_edge(part_id, model_id, relation='used_in')
    for _, row in suppliers.iterrows():
        G.add_node(row['supplier_id'], type='supplier', tier=row['tier'], label=row['name'])
        G.add_edge(row['supplier_id'], row['part_id'], relation='supplies')
    for downstream, upstream in supplier_feeds:
        G.add_edge(upstream, downstream, relation='feeds')
    return G

G = build_graph(connector, plants, models, bom, parts)

def trace_impact(graph, disrupted_supplier_id):
    affected = {'suppliers': set(), 'parts': set(), 'models': set(), 'plants': set()}
    frontier, visited = [disrupted_supplier_id], set()
    while frontier:
        node = frontier.pop()
        if node in visited:
            continue
        visited.add(node)
        t = graph.nodes[node].get('type')
        if t == 'supplier': affected['suppliers'].add(node)
        elif t == 'part': affected['parts'].add(node)
        elif t == 'model': affected['models'].add(node)
        elif t == 'plant': affected['plants'].add(node)
        frontier.extend(graph.successors(node))
    return affected

print(f'Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.')


Graph built: 19 nodes, 26 edges.


## 5. Dynamic agents

The key change from v1: agents no longer return fixed numbers. They query the connector and **score real candidate options**, so the same code produces different recommendations depending on what the connector returns — synthetic today, a live SAP/Oracle/Zoho feed later.

In [6]:
@dataclass
class AgentProposal:
    agent: str
    option: str
    cost_delta_pct: float
    delay_days: float
    confidence: float
    risk_note: str
    buffer_days: Optional[float] = None  # set by InventoryAgent only; other agents leave this as None


class ProcurementAgent:
    name = 'Procurement Agent (Source)'

    def __init__(self, connector: SCMConnector):
        self.connector = connector

    def propose(self, part_id: str, excluded_supplier_ids: set) -> AgentProposal:
        candidates = self.connector.get_suppliers(part_id)
        if candidates.empty or 'supplier_id' not in candidates.columns:
            return AgentProposal(self.name, 'No alternate supplier available', 0, 999, 0.0,
                                  'High risk: no supplier data returned for this part')
        candidates = candidates[~candidates['supplier_id'].isin(excluded_supplier_ids)]
        if candidates.empty:
            return AgentProposal(self.name, 'No alternate supplier available', 0, 999, 0.0, 'High risk: single source exhausted')
        # Dynamic scoring: reliability and financial health favored, cost and lead time penalized.
        # Guarded against an all-zero cost or lead-time column (e.g. a live connector that
        # hasn't wired in real cost data yet) so the score never silently becomes NaN.
        candidates = candidates.copy()
        lead_max = candidates['lead_time_days'].max()
        cost_max = candidates['unit_cost_usd'].max()
        candidates['score'] = (
            candidates['reliability'] * 0.4
            + candidates['financial_health'] * 0.3
            - (candidates['lead_time_days'] / lead_max if lead_max else 0) * 0.2
            - (candidates['unit_cost_usd'] / cost_max if cost_max else 0) * 0.1
        )
        best = candidates.sort_values('score', ascending=False).iloc[0]
        baseline_cost = candidates['unit_cost_usd'].min()
        cost_delta_pct = ((best['unit_cost_usd'] - baseline_cost) / baseline_cost) * 100 if baseline_cost else 0
        requal_days = 14 if not best['certified'] else 7
        return AgentProposal(
            agent=self.name,
            option=f"Switch to {best['name']} (Tier {best['tier']}, {best['region']})",
            cost_delta_pct=round(cost_delta_pct, 1),
            delay_days=requal_days,
            confidence=round(best['reliability'], 2),
            risk_note=f"Selected by score across {len(candidates)} candidates -- reliability {best['reliability']}, lead time {best['lead_time_days']}d"
        )


class InventoryAgent:
    name = 'Inventory Agent (Plan)'

    def __init__(self, connector: SCMConnector):
        self.connector = connector

    def propose(self, part_id: str, plant_id: str, models_using_part: List[str]) -> AgentProposal:
        on_hand = self.connector.get_inventory(part_id, plant_id)
        daily_consumption = sum(self.connector.get_avg_weekly_demand(m) for m in models_using_part) / 7
        buffer_days = round(on_hand / daily_consumption, 1) if daily_consumption else 999
        return AgentProposal(
            agent=self.name,
            option=f'Draw down existing buffer stock ({on_hand} units on hand)',
            cost_delta_pct=0.0,
            delay_days=0,
            confidence=0.95,
            risk_note=f'Computed from live inventory/demand: buffer covers {buffer_days} days before line impact',
            buffer_days=buffer_days,
        )


class ProductionAgent:
    name = 'Production Agent (Make)'

    def __init__(self, connector: SCMConnector):
        self.connector = connector

    def propose(self, affected_models: set, plant_id: str) -> AgentProposal:
        capacity = self.connector.get_plant_capacity(plant_id)
        affected_demand = sum(self.connector.get_avg_weekly_demand(m) for m in affected_models)
        exposure_pct = min(1.0, affected_demand / capacity) if capacity else 1.0
        delay_days = round(exposure_pct * 10, 1)  # more exposure -> longer slowdown needed
        return AgentProposal(
            agent=self.name,
            option=f'Slow production on {len(affected_models)} affected model line(s), reallocate freed capacity',
            cost_delta_pct=round(exposure_pct * 6, 1),
            delay_days=delay_days,
            confidence=0.8,
            risk_note=f'Affected demand is {exposure_pct*100:.0f}% of plant weekly capacity'
        )


class LogisticsAgent:
    name = 'Logistics Agent (Deliver)'

    def __init__(self, connector: SCMConnector):
        self.connector = connector

    def propose(self, days_available_before_stockout: float) -> AgentProposal:
        options = self.connector.get_logistics_options()
        # Pick the cheapest mode that still lands within the available window
        feasible = options[options['days_transit'] <= max(days_available_before_stockout, 1)]
        chosen = feasible.sort_values('cost_index').iloc[0] if not feasible.empty else options.sort_values('days_transit').iloc[0]
        baseline = options.sort_values('cost_index').iloc[0]
        time_saved = baseline['days_transit'] - chosen['days_transit']
        cost_delta_pct = (chosen['cost_index'] - baseline['cost_index']) / baseline['cost_index'] * 100
        return AgentProposal(
            agent=self.name,
            option=f"Ship via {chosen['mode']} ({chosen['days_transit']}d transit)",
            cost_delta_pct=round(cost_delta_pct, 1),
            delay_days=round(-time_saved, 1),
            confidence=0.85,
            risk_note=f'Chosen from {len(options)} carrier modes to fit a {days_available_before_stockout:.1f}-day window'
        )

print('Dynamic agents defined.')


Dynamic agents defined.


## 6. Orchestrator with configurable policy weights

Weights are passed in as config, not hardcoded — mirroring how SAP IBP / Oracle SCM Cloud expose tunable planning parameters rather than baking priorities into code. Swap `policy` per business unit or region without touching agent logic.

In [7]:
class OrchestratorAgent:
    name = 'Orchestrator'

    def __init__(self, connector: SCMConnector, policy: dict):
        self.connector = connector
        self.policy = policy  # e.g. {'cost_weight': 1.0, 'time_weight': 1.5}

    def negotiate(self, proposals: List[AgentProposal]) -> dict:
        trace = ['--- Negotiation trace ---']
        total_cost, total_delay, weighted_confidence = 0.0, 0.0, []
        for p in proposals:
            trace.append(f'[{p.agent}] proposes: {p.option}')
            trace.append(f"    -> cost {p.cost_delta_pct:+.1f}% | time {p.delay_days:+.1f}d | confidence {p.confidence:.2f} | {p.risk_note}")
            total_cost += p.cost_delta_pct
            total_delay += p.delay_days
            weighted_confidence.append(p.confidence)
        score = self.policy['cost_weight'] * total_cost + self.policy['time_weight'] * total_delay
        avg_confidence = round(sum(weighted_confidence) / len(weighted_confidence), 2) if weighted_confidence else 0
        trace += [
            '---',
            f'Combined plan: total cost impact {total_cost:+.1f}%, total schedule impact {total_delay:+.1f} days',
            f"Weighted score: {score:.1f} (policy weights -- cost x{self.policy['cost_weight']}, time x{self.policy['time_weight']})",
            f'Average confidence across agents: {avg_confidence}',
            '---',
            'FINAL RECOMMENDATION: proceed with combined plan above, pending manager approval.'
        ]
        decision = {
            'action_type': 'supply_chain_response_plan',
            'summary': f'cost {total_cost:+.1f}%, schedule {total_delay:+.1f}d, score {score:.1f}',
            'total_cost_pct': total_cost,
            'total_delay_days': total_delay,
            'score': score,
            'avg_confidence': avg_confidence,
            'trace': trace,
        }
        return decision

print('Orchestrator ready.')

Orchestrator ready.


## 7. Run two disruption scenarios through the same pipeline

The point of this section is to show the logic **generalizes** — the same connector, agents, and orchestrator run different real-world disruption types without any code changes, only different input events.

In [8]:
def run_scenario(scenario_name, disrupted_supplier_id, plant_id, policy):
    impact = trace_impact(G, disrupted_supplier_id)
    part_id = list(impact['parts'])[0]
    affected_models = impact['models']
    excluded = impact['suppliers']

    proc = ProcurementAgent(connector).propose(part_id, excluded)
    inv = InventoryAgent(connector).propose(part_id, plant_id, list(affected_models))
    prod = ProductionAgent(connector).propose(affected_models, plant_id)

    buffer_days_hint = inv.buffer_days if inv.buffer_days is not None else 999
    log = LogisticsAgent(connector).propose(buffer_days_hint)

    orchestrator = OrchestratorAgent(connector, policy)
    decision = orchestrator.negotiate([proc, inv, prod, log])

    render_agent_report(
        title=f'Scenario: {scenario_name}',
        subtitle=(f'Disrupted supplier: {disrupted_supplier_id}  →  affected part: {part_id}  →  '
                  f'{len(affected_models)} model(s) impacted'),
        proposals=[proc, inv, prod, log],
        decision=decision,
    )
    connector.post_decision(decision)
    return decision

policy_cost_sensitive = {'cost_weight': 1.5, 'time_weight': 1.0}
policy_time_sensitive = {'cost_weight': 1.0, 'time_weight': 2.0}

decision_1 = run_scenario('Tier-3 wafer fab slowdown (ECU chip)', 'SUP_003', 'PLANT_A', policy_time_sensitive)
decision_2 = run_scenario('Tier-1 battery supplier disruption', 'SUP_007', 'PLANT_B', policy_cost_sensitive)


Agent,Recommendation,Cost Impact,Time Impact,Confidence,Why
Procurement Agent (Source),"Switch to Quarzon Materials (Tier 3, Germany)",+0.0%,+7.0 days,82%,"Selected by score across 2 candidates -- reliability 0.82, lead time 40d"
Inventory Agent (Plan),Draw down existing buffer stock (5400 units on hand),+0.0%,+0.0 days,95%,Computed from live inventory/demand: buffer covers 16.8 days before line impact
Production Agent (Make),"Slow production on 3 affected model line(s), reallocate freed capacity",+3.2%,+5.3 days,80%,Affected demand is 53% of plant weekly capacity
Logistics Agent (Deliver),Ship via rail (14d transit),+35.0%,-10.0 days,85%,Chosen from 3 carrier modes to fit a 16.8-day window


[connector] would write back to source system: supply_chain_response_plan -> cost +38.2%, schedule +2.3d, score 42.8


Agent,Recommendation,Cost Impact,Time Impact,Confidence,Why
Procurement Agent (Source),"Switch to Ionix Power (Tier 1, Sweden)",+0.0%,+7.0 days,83%,"Selected by score across 1 candidates -- reliability 0.83, lead time 22d"
Inventory Agent (Plan),Draw down existing buffer stock (1900 units on hand),+0.0%,+0.0 days,95%,Computed from live inventory/demand: buffer covers 19.3 days before line impact
Production Agent (Make),"Slow production on 1 affected model line(s), reallocate freed capacity",+1.3%,+2.2 days,80%,Affected demand is 22% of plant weekly capacity
Logistics Agent (Deliver),Ship via rail (14d transit),+35.0%,-10.0 days,85%,Chosen from 3 carrier modes to fit a 19.3-day window


[connector] would write back to source system: supply_chain_response_plan -> cost +36.3%, schedule -0.8d, score 53.6


## 8. Compare outcomes under different policies

Same disruption, different business priority (cost-sensitive vs. time-sensitive policy) — this is the kind of trade-off a real supply chain manager configures per region or per part criticality in SAP IBP / Oracle SCM Cloud today.

In [9]:
comparison = pd.DataFrame([
    {'scenario': 'Wafer fab slowdown (time-sensitive policy)', 'cost_%': decision_1['total_cost_pct'], 'delay_days': decision_1['total_delay_days'], 'score': decision_1['score']},
    {'scenario': 'Battery supplier disruption (cost-sensitive policy)', 'cost_%': decision_2['total_cost_pct'], 'delay_days': decision_2['total_delay_days'], 'score': decision_2['score']},
])

display(HTML('<h4 style="margin:12px 0 6px 0;">Policy comparison — same disruptions, different priorities</h4>'))
display(
    comparison.style
        .format({'cost_%': '{:+.1f}%', 'delay_days': '{:+.1f} days', 'score': '{:.1f}'})
        .background_gradient(subset=['score'], cmap='RdYlGn_r')
)


,scenario,cost_%,delay_days,score
0,Wafer fab slowdown (time-sensitive policy),+38.2%,+2.3 days,42.8
1,Battery supplier disruption (cost-sensitive policy),+36.3%,-0.8 days,53.6


## 9. Live Zoho Inventory connector

One connector class, credentials only from Colab Secrets. This is the consolidated, working version — everything the agents need, nothing hardcoded.

In [10]:
from google.colab import userdata
import requests
import pandas as pd
from typing import List

# Fixed column set for supplier data, so a live connector always returns a
# properly-shaped DataFrame even when zero vendors come back from Zoho --
# a bare `pd.DataFrame([])` has no columns at all, which used to crash the
# very next line that filters on 'supplier_id'.
_SUPPLIER_COLUMNS = ['part_id', 'tier', 'name', 'region', 'reliability', 'financial_health',
                     'lead_time_days', 'unit_cost_usd', 'weekly_capacity', 'certified', 'supplier_id']


class ZohoConnector(SCMConnector):
    """Live Zoho Inventory connector. Region defaults to India (.in) -- change accounts_url/api_base if your org is on a different data center."""

    def __init__(self, region='in'):
        self.client_id = userdata.get('ZOHO_CLIENT_ID')
        self.client_secret = userdata.get('ZOHO_CLIENT_SECRET')
        self.refresh_token = userdata.get('ZOHO_REFRESH_TOKEN')
        self.org_id = userdata.get('ZOHO_ORG_ID')
        self.accounts_url = f'https://accounts.zoho.{region}'
        self.api_base = f'https://www.zohoapis.{region}/inventory/v1'
        self._access_token = None

    def _get_access_token(self):
        if self._access_token:
            return self._access_token
        try:
            r = requests.post(f'{self.accounts_url}/oauth/v2/token', data={
                'refresh_token': self.refresh_token,
                'client_id': self.client_id,
                'client_secret': self.client_secret,
                'grant_type': 'refresh_token',
            })
            r.raise_for_status()
            self._access_token = r.json()['access_token']
        except Exception as e:
            raise RuntimeError(
                'Could not authenticate with Zoho. Double-check ZOHO_CLIENT_ID, ZOHO_CLIENT_SECRET, '
                f'and ZOHO_REFRESH_TOKEN in Colab Secrets. Original error: {e}'
            )
        return self._access_token

    def _headers(self):
        return {'Authorization': f'Zoho-oauthtoken {self._get_access_token()}'}

    def _get(self, endpoint, params=None):
        params = dict(params or {})
        params['organization_id'] = self.org_id
        r = requests.get(f'{self.api_base}/{endpoint}', headers=self._headers(), params=params)
        r.raise_for_status()
        return r.json()

    def _post(self, endpoint, body):
        r = requests.post(f'{self.api_base}/{endpoint}', headers=self._headers(),
                           params={'organization_id': self.org_id}, json=body)
        r.raise_for_status()
        return r.json()

    # ---- cached item list so a full scan doesn't re-fetch per part ----
    def all_items(self):
        if not hasattr(self, '_items_cache'):
            try:
                self._items_cache = self._get('items').get('items', [])
            except Exception as e:
                print(f'  [ZohoConnector] could not fetch items, treating inventory as empty: {e}')
                self._items_cache = []
        return self._items_cache

    # ---- SCMConnector interface ----
    def get_suppliers(self, part_id: str) -> pd.DataFrame:
        try:
            vendors = self._get('contacts', {'contact_type': 'vendor'}).get('contacts', [])
        except Exception as e:
            print(f'  [ZohoConnector] could not fetch vendors for {part_id}, returning no candidates: {e}')
            vendors = []
        if not vendors:
            return pd.DataFrame(columns=_SUPPLIER_COLUMNS)
        rows = [{
            'part_id': part_id, 'tier': 1,
            'name': v.get('contact_name', 'Unknown'),
            'region': (v.get('billing_address') or {}).get('country', 'Unknown'),
            'reliability': 0.85, 'financial_health': 0.80, 'lead_time_days': 14,
            'unit_cost_usd': 0.0, 'weekly_capacity': 5000, 'certified': True,
            'supplier_id': str(v.get('contact_id', f'V{i}')),
        } for i, v in enumerate(vendors)]
        return pd.DataFrame(rows, columns=_SUPPLIER_COLUMNS)

    def get_supplier_upstream(self, supplier_id: str) -> List[str]:
        return []  # Zoho has no multi-tier supplier concept

    def get_inventory(self, part_id: str, plant_id: str) -> int:
        for item in self.all_items():
            item_text = ((item.get('name') or '') + (item.get('sku') or '')).upper()
            if part_id.upper() in item_text:
                return int(float(item.get('stock_on_hand', 0)))
        return 0

    def get_avg_weekly_demand(self, model_id: str) -> float:
        return 100.0  # Zoho has no model-level demand history -- safe default until sales-order history is wired in

    def get_plant_capacity(self, plant_id: str) -> int:
        return 5000  # Zoho has no plant/capacity concept -- default until a side config table is added

    def get_logistics_options(self) -> pd.DataFrame:
        return pd.DataFrame([
            {'mode': 'sea', 'days_transit': 24, 'cost_index': 1.00},
            {'mode': 'rail', 'days_transit': 14, 'cost_index': 1.35},
            {'mode': 'air', 'days_transit': 5, 'cost_index': 2.60},
        ])

    def post_decision(self, decision: dict) -> None:
        try:
            body = {'vendor_id': decision.get('vendor_id', ''), 'date': decision.get('date', ''),
                    'line_items': decision.get('line_items', [{'name': 'SCM Agent recommended item', 'quantity': 1, 'rate': 1.0}])}
            result = self._post('purchaseorders', body)
            print(f"[Zoho] Purchase order created -> {result.get('purchaseorder', {}).get('purchaseorder_id')}")
        except Exception as e:
            print(f'[Zoho] post_decision not completed (expected until real vendor_id/line_items are supplied): {e}')


print('ZohoConnector defined -- credentials read from Colab Secrets only.')


ZohoConnector defined -- credentials read from Colab Secrets only.


## 10. Automatic scan — no manual part_id needed

`auto_scan_and_respond()` checks every part you care about against **live** Zoho stock, computes buffer days from real numbers, and only runs the full four-agent negotiation for parts that actually need it (buffer below your threshold). Parts that are fine are reported as fine, in one line, without wasting a negotiation on them. This is what makes it automatic: you set a threshold once, not a part_id every time.

In [13]:
PART_TO_MODELS = {
    'PART_ECU': ['MODEL_SUV1', 'MODEL_SED1', 'MODEL_SUV2'],
    'PART_SEAT': ['MODEL_SUV1', 'MODEL_SED1'],
    'PART_BATT': ['MODEL_SUV2'],
    'PART_TIRE': ['MODEL_SUV1', 'MODEL_SED1', 'MODEL_SUV2'],
}
def auto_scan_and_respond(connector, part_to_models=PART_TO_MODELS, plant_id='PLANT_A',
                           buffer_threshold_days=10.0, policy=None):
    """Scans every part in part_to_models against live inventory. Runs the full agent
    negotiation automatically for any part whose computed buffer is below the threshold.
    Returns a summary DataFrame across everything scanned."""
    if policy is None:
        policy = {'cost_weight': 1.0, 'time_weight': 1.5}

    display(HTML(f'''
        <div style="background:#2c3e50;color:white;padding:14px 18px;border-radius:8px;
                    font-family:-apple-system,Segoe UI,Roboto,sans-serif;margin-bottom:6px;">
          <h2 style="margin:0;">Automatic Supply Chain Risk Scan</h2>
          <p style="margin:4px 0 0 0;color:#dfe6e9;">Live Zoho Inventory &middot; flagging any part with less than
             {buffer_threshold_days} days of buffer stock</p>
        </div>
    '''))

    summary_rows = []

    for part_id, models in part_to_models.items():
        on_hand = connector.get_inventory(part_id, plant_id)
        daily_consumption = sum(connector.get_avg_weekly_demand(m) for m in models) / 7
        buffer_days = round(on_hand / daily_consumption, 1) if daily_consumption else 999
        at_risk = buffer_days < buffer_threshold_days

        status_label = 'AT RISK' if at_risk else 'OK'
        action_note = '-> running full analysis below' if at_risk else '-> no action needed'
        print(f'  [{status_label:<7}] {part_id:<12} {on_hand:>7,} units on hand  ->  {buffer_days:>6.1f} days of buffer  {action_note}')

        if not at_risk:
            summary_rows.append({'part_id': part_id, 'buffer_days': buffer_days, 'status': 'OK',
                                  'recommendation': 'No action needed', 'cost_impact_pct': 0.0, 'schedule_impact_days': 0.0})
            continue

        proc = ProcurementAgent(connector).propose(part_id, set())
        inv = InventoryAgent(connector).propose(part_id, plant_id, models)
        prod = ProductionAgent(connector).propose(set(models), plant_id)
        log = LogisticsAgent(connector).propose(buffer_days)
        orchestrator = OrchestratorAgent(connector, policy)
        decision = orchestrator.negotiate([proc, inv, prod, log])

        print_readable_report(part_id, on_hand, buffer_days, [proc, inv, prod, log], decision)

        summary_rows.append({
            'part_id': part_id, 'buffer_days': buffer_days, 'status': 'AT RISK',
            'recommendation': proc.option,
            'cost_impact_pct': decision['total_cost_pct'],
            'schedule_impact_days': decision['total_delay_days'],
        })

    return pd.DataFrame(summary_rows)

print('auto_scan_and_respond() defined.')


auto_scan_and_respond() defined.


## 11. Readable report formatting

This turns the raw agent objects and orchestrator trace into plain-English output -- the same style you already built and tested, kept here as one shared function so every scan uses it consistently.

In [14]:
def print_readable_report(part_id, on_hand, buffer_days, proposals, decision):
    """Renders a clear, color-coded report card for one at-risk part
    (a plain-English recommendation, not a raw trace dump)."""
    render_agent_report(
        title=f'Disruption Response: {part_id}',
        subtitle=f'{on_hand:,} units on hand — {buffer_days} days of buffer remaining',
        proposals=proposals,
        decision=decision,
    )

print('print_readable_report() defined.')


print_readable_report() defined.


## 12. Run the automatic scan

One call. No part_id argument needed -- it checks everything in `PART_TO_MODELS` against live Zoho data and only stops to negotiate a response for parts that actually need it.

In [ ]:
live_connector = ZohoConnector()
results = auto_scan_and_respond(live_connector, buffer_threshold_days=10.0)

display(HTML('<h3 style="margin:18px 0 6px 0;">Scan Summary — All Tracked Parts</h3>'))
display(style_scan_results(results))


## 13. Honest state of this integration

**Real and tested (by you, against your own Zoho account):** vendor list, item stock levels, OAuth token refresh, and the automatic buffer-day scan across multiple parts.

**Still default/synthetic, and worth saying plainly if you present this:**
- `get_avg_weekly_demand` returns a fixed 100 units/week for every model -- Zoho has no model-level demand history, so buffer-day numbers above are only as accurate as that placeholder. Wiring this to real Zoho sales-order history is the next concrete step.
- `get_plant_capacity` and `get_logistics_options` are still fixed defaults -- Zoho has no native concept of either.
- `post_decision` will fail gracefully (caught and printed, not crashing) until it's given a real `vendor_id` and real `line_items` -- right now it's a proof that the write-back path exists, not a finished purchase-order workflow.
- `PART_TO_MODELS` is a hand-written mapping, not pulled from Zoho -- if you add or rename items in Zoho, update this dictionary to match.

## 14. Accuracy upgrade — real demand, real cost (fixing a bug, not just adding a feature)

**Bug being fixed:** in the v3 `ZohoConnector`, `get_avg_weekly_demand` always returned a flat `100.0` and every vendor's `unit_cost_usd` was hardcoded to `0.0`. That second one is more serious than it looks — the Procurement Agent's scoring formula subtracts a cost penalty, but if every candidate has the same cost (`0.0`), that term is always zero and **cost silently never influenced which supplier got picked.** The core negotiation was running on three real signals and one fake tie. `ZohoConnectorPro` below fixes both:

- `get_avg_weekly_demand` now aggregates real Zoho **sales order history** over a lookback window instead of a placeholder
- `get_suppliers` now reads each item's real **purchase rate** from Zoho instead of `0.0`, so Procurement's cost term is real
- vendor **reliability** is now estimated from real purchase-order status history where it exists (fraction closed/received vs. cancelled/pending), falling back to a clearly-labeled default only when a vendor has no PO history yet

**Being upfront about the remaining limits:** Zoho still has no native per-vendor part pricing (cost reflects the item's rate, not what each specific vendor charges for it, since that needs PO-level price history per vendor) and no multi-tier supplier concept. Those are structural gaps in Zoho's data model, not something more code can fix without a second data source.

In [ ]:
from datetime import datetime, timedelta
import time


class ZohoConnectorPro(ZohoConnector):
    """Same interface, higher-accuracy implementations backed by real sales and purchase history."""

    def __init__(self, region='in', demand_lookback_weeks=12, max_orders_scanned=100):
        super().__init__(region=region)
        self.demand_lookback_weeks = demand_lookback_weeks
        self.max_orders_scanned = max_orders_scanned
        self._demand_cache = {}
        self._reliability_cache = {}

    def get_avg_weekly_demand(self, model_id: str) -> float:
        if model_id in self._demand_cache:
            return self._demand_cache[model_id]
        cutoff = (datetime.now() - timedelta(weeks=self.demand_lookback_weeks)).strftime('%Y-%m-%d')
        try:
            orders = self._get('salesorders', {'date_start': cutoff}).get('salesorders', [])
        except Exception as e:
            print(f'  [ZohoConnectorPro] sales order fetch failed, falling back to default demand for {model_id}: {e}')
            self._demand_cache[model_id] = 100.0
            return 100.0

        total_units = 0.0
        scanned = 0
        for order in orders[:self.max_orders_scanned]:
            scanned += 1
            line_items = order.get('line_items')
            if line_items is None:
                # list endpoint didn't include line items -- fetch the order detail
                try:
                    detail = self._get(f"salesorders/{order['salesorder_id']}")
                    line_items = detail.get('salesorder', {}).get('line_items', [])
                except Exception:
                    continue
            for li in line_items:
                # Zoho sometimes returns an explicit null for a missing field rather than
                # omitting the key, so `.get(key, '')` alone can still hand back None here --
                # guard each side with `or ''` before concatenating.
                item_name = ((li.get('name') or '') + (li.get('sku') or '')).upper()
                if model_id.upper() in item_name:
                    total_units += float(li.get('quantity', 0))

        weekly = round(total_units / self.demand_lookback_weeks, 1) if scanned else 100.0
        result = weekly if weekly > 0 else 100.0  # never divide-by-zero downstream on a genuine no-sales-found case
        print(f'  [ZohoConnectorPro] {model_id}: {total_units:.0f} units across {scanned} orders '
              f'over {self.demand_lookback_weeks}w -> {result} units/week')
        self._demand_cache[model_id] = result
        return result

    def _estimate_vendor_reliability(self, vendor_id: str) -> float:
        if vendor_id in self._reliability_cache:
            return self._reliability_cache[vendor_id]
        try:
            pos = self._get('purchaseorders', {'vendor_id': vendor_id}).get('purchaseorders', [])
        except Exception:
            pos = []
        if not pos:
            self._reliability_cache[vendor_id] = 0.75  # neutral default, explicitly lower than a proven-good vendor -- no history to trust yet
            return 0.75
        good_statuses = {'closed', 'received', 'billed'}
        good = sum(1 for po in pos if str(po.get('status', '')).lower() in good_statuses)
        score = round(0.5 + 0.5 * (good / len(pos)), 2)  # floor at 0.5 so one bad batch doesn't zero out a vendor
        self._reliability_cache[vendor_id] = score
        return score

    def get_suppliers(self, part_id: str) -> pd.DataFrame:
        base = super().get_suppliers(part_id)
        if base.empty:
            return base
        real_cost = 0.0
        for item in self.all_items():
            item_text = ((item.get('name') or '') + (item.get('sku') or '')).upper()
            if part_id.upper() in item_text:
                real_cost = float(item.get('purchase_rate', 0) or item.get('rate', 0) or 0)
                break
        base = base.copy()
        base['unit_cost_usd'] = real_cost if real_cost > 0 else base['unit_cost_usd']
        base['reliability'] = base['supplier_id'].apply(self._estimate_vendor_reliability)
        return base


print('ZohoConnectorPro defined -- real demand, real cost, PO-history-based reliability.')


## 17. Fully dynamic item discovery -- no hardcoded part list

`PART_TO_MODELS` (Section 10) was a fixed dict of part IDs you had to maintain by hand. `discover_tracked_items()` below replaces it: it asks Zoho for every active item and pulls its name, SKU, on-hand stock and (if set) its linked vendor directly from the API. Add a new item in Zoho and it is picked up automatically on the next scan -- nothing in this notebook needs to change.


In [ ]:
def _discover_tracked_items(self):
    """Pulls every active item from Zoho Inventory dynamically -- item name, SKU,
    on-hand stock and linked vendor all come live from the API. Replaces the fixed
    PART_TO_MODELS dict: nothing here is hardcoded, so a new item in Zoho is picked
    up automatically on the next call, with no notebook edits."""
    rows = []
    for it in self.all_items():
        if str(it.get('status', 'active')).lower() != 'active':
            continue
        rows.append({
            'item_id': str(it.get('item_id', '')),
            'sku': it.get('sku') or it.get('name') or str(it.get('item_id', '')),
            'name': it.get('name', 'Unknown item'),
            'stock_on_hand': float(it.get('stock_on_hand', 0) or 0),
            'reorder_level': float(it.get('reorder_level', 0) or 0),
            'vendor_id': str(it.get('vendor_id') or ''),
            'vendor_name': it.get('vendor_name') or '',
        })
    df = pd.DataFrame(rows)
    if df.empty:
        print('  [ZohoConnectorPro] no active items returned from Zoho -- nothing to scan.')
    return df

# Attached onto the class rather than re-editing Section 14's class body, so the
# existing ZohoConnectorPro definition above stays the single source of truth.
ZohoConnectorPro.discover_tracked_items = _discover_tracked_items
print('discover_tracked_items() attached -- item list is now fully dynamic.')


## 18. Dynamic scan -- item names and count come from Zoho, not from code

`auto_scan_and_respond_dynamic()` is the drop-in replacement for `auto_scan_and_respond()`. Instead of looping over a fixed dict, it loops over whatever `discover_tracked_items()` returns this run. Each item's SKU is passed straight into the existing connector methods (`get_inventory`, `get_avg_weekly_demand`) exactly the way a model/part ID was before -- the agents in Section 5 are unchanged.


In [ ]:
def auto_scan_and_respond_dynamic(connector, plant_id='PLANT_A', buffer_threshold_days=10.0,
                                   policy=None, max_items=None):
    """Fully dynamic replacement for auto_scan_and_respond(): the item list, item
    names and item count all come live from Zoho via discover_tracked_items() --
    no PART_TO_MODELS dict required. Everything downstream (agents, orchestrator,
    report rendering) is reused unchanged."""
    if policy is None:
        policy = {'cost_weight': 1.0, 'time_weight': 1.5}

    items_df = connector.discover_tracked_items()
    if items_df.empty:
        print('No active items found in Zoho Inventory -- nothing to scan.')
        return pd.DataFrame()
    if max_items:
        items_df = items_df.head(max_items)

    display(HTML(f'''
        <div style="background:#2c3e50;color:white;padding:14px 18px;border-radius:8px;
                    font-family:-apple-system,Segoe UI,Roboto,sans-serif;margin-bottom:6px;">
          <h2 style="margin:0;">Automatic Supply Chain Risk Scan (Dynamic)</h2>
          <p style="margin:4px 0 0 0;color:#dfe6e9;">Live Zoho Inventory &middot; {len(items_df)} items discovered this run &middot; flagging anything with less than {buffer_threshold_days} days of buffer stock</p>
        </div>
    '''))

    summary_rows = []
    for _, item in items_df.iterrows():
        part_id, item_name = item['sku'], item['name']
        on_hand = connector.get_inventory(part_id, plant_id)
        daily_consumption = connector.get_avg_weekly_demand(part_id) / 7
        buffer_days = round(on_hand / daily_consumption, 1) if daily_consumption else 999
        at_risk = buffer_days < buffer_threshold_days

        status_label = 'AT RISK' if at_risk else 'OK'
        action_note = '-> running full analysis below' if at_risk else '-> no action needed'
        print(f'  [{status_label:<7}] {str(item_name)[:28]:<28} {on_hand:>7,.0f} units  ->  '
              f'{buffer_days:>6.1f}d buffer  {action_note}')

        if not at_risk:
            summary_rows.append({'item': item_name, 'sku': part_id, 'buffer_days': buffer_days,
                                  'status': 'OK', 'recommendation': 'No action needed',
                                  'cost_impact_pct': 0.0, 'schedule_impact_days': 0.0})
            continue

        proc = ProcurementAgent(connector).propose(part_id, set())
        inv = InventoryAgent(connector).propose(part_id, plant_id, [part_id])
        prod = ProductionAgent(connector).propose({part_id}, plant_id)
        log = LogisticsAgent(connector).propose(buffer_days)
        orchestrator = OrchestratorAgent(connector, policy)
        decision = orchestrator.negotiate([proc, inv, prod, log])

        print_readable_report(part_id, on_hand, buffer_days, [proc, inv, prod, log], decision)

        summary_rows.append({
            'item': item_name, 'sku': part_id, 'buffer_days': buffer_days, 'status': 'AT RISK',
            'recommendation': proc.option,
            'cost_impact_pct': decision['total_cost_pct'],
            'schedule_impact_days': decision['total_delay_days'],
        })

    return pd.DataFrame(summary_rows)

print('auto_scan_and_respond_dynamic() defined -- no hardcoded part list required.')


## 19. Vendor analysis -- built from real Zoho purchase-order history

`analyze_vendors()` pulls every vendor Zoho knows about and, for each one, scores them on what actually happened in their purchase-order history: order volume, total spend, on-time delivery rate, average lead time, and a composite reliability score. It also flags spend concentration risk (too much of your spend sitting with one vendor). Nothing here is hardcoded -- add a vendor in Zoho and it appears on the next run.


In [ ]:
def analyze_vendors(connector, lookback_weeks=12):
    """Builds a vendor scorecard entirely from live Zoho data: every vendor in
    Zoho Contacts, scored against their real purchase-order history over the
    lookback window. No vendor list or vendor names are hardcoded."""
    try:
        vendors = connector._get('contacts', {'contact_type': 'vendor'}).get('contacts', [])
    except Exception as e:
        print(f'Could not fetch vendors from Zoho: {e}')
        return pd.DataFrame()

    cutoff = (datetime.now() - timedelta(weeks=lookback_weeks)).strftime('%Y-%m-%d')
    good_statuses = {'closed', 'received', 'billed'}
    rows = []
    for v in vendors:
        vendor_id = str(v.get('contact_id', ''))
        vendor_name = v.get('contact_name', 'Unknown vendor')
        try:
            pos = connector._get('purchaseorders', {'vendor_id': vendor_id, 'date_start': cutoff}).get('purchaseorders', [])
        except Exception:
            pos = []

        if not pos:
            rows.append({'vendor_id': vendor_id, 'vendor_name': vendor_name, 'orders': 0,
                         'total_spend_usd': 0.0, 'on_time_rate': None, 'avg_lead_time_days': None,
                         'reliability_score': 0.5, 'note': f'No PO history in last {lookback_weeks}w'})
            continue

        total_spend = sum(float(po.get('total', 0) or 0) for po in pos)
        on_time = sum(1 for po in pos if str(po.get('status', '')).lower() in good_statuses)
        on_time_rate = round(on_time / len(pos), 2)

        lead_times = []
        for po in pos:
            od, dd = po.get('date'), po.get('delivery_date') or po.get('expected_delivery_date')
            if od and dd:
                try:
                    lead_times.append((datetime.strptime(dd, '%Y-%m-%d') - datetime.strptime(od, '%Y-%m-%d')).days)
                except Exception:
                    continue
        avg_lead = round(sum(lead_times) / len(lead_times), 1) if lead_times else None

        # Floor at 0.5 so a vendor with limited history isn't unfairly zeroed out --
        # mirrors the same floor used for supplier reliability in Section 14.
        reliability = round(0.5 + 0.5 * on_time_rate, 2)
        rows.append({'vendor_id': vendor_id, 'vendor_name': vendor_name, 'orders': len(pos),
                     'total_spend_usd': round(total_spend, 2), 'on_time_rate': on_time_rate,
                     'avg_lead_time_days': avg_lead, 'reliability_score': reliability, 'note': ''})

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values('total_spend_usd', ascending=False).reset_index(drop=True)
        total = df['total_spend_usd'].sum()
        df['spend_share_pct'] = round(df['total_spend_usd'] / total * 100, 1) if total else 0.0
    return df


def render_vendor_scorecard(vendor_df, concentration_threshold_pct=50.0):
    """Renders the vendor scorecard and calls out concentration risk (too much
    spend riding on one vendor) and low-reliability vendors."""
    display(HTML('<h3 style="margin:18px 0 6px 0;">Vendor Analysis -- Live Zoho Purchase History</h3>'))
    if vendor_df.empty:
        print('No vendor data available from Zoho.')
        return

    display(
        vendor_df.style
            .format({'total_spend_usd': '${:,.2f}', 'spend_share_pct': '{:.1f}%',
                      'on_time_rate': '{:.0%}', 'reliability_score': '{:.2f}'}, na_rep='n/a')
            .background_gradient(subset=['reliability_score'], cmap='RdYlGn', vmin=0.5, vmax=1.0)
    )

    top = vendor_df.iloc[0]
    if top['spend_share_pct'] >= concentration_threshold_pct:
        print(f"\n\u26a0 Concentration risk: {top['vendor_name']} accounts for "
              f"{top['spend_share_pct']}% of tracked spend -- a disruption there hits hardest.")

    at_risk_vendors = vendor_df[vendor_df['reliability_score'] < 0.7]
    if not at_risk_vendors.empty:
        names = ', '.join(at_risk_vendors['vendor_name'].tolist())
        print(f"\u26a0 Below-threshold reliability (<0.70): {names}")

print('analyze_vendors() and render_vendor_scorecard() defined.')


## 20. Run the dynamic scan + vendor analysis together

One call for stock risk across every live item, one call for vendor performance -- both driven entirely by what Zoho returns right now.


In [ ]:
live_connector_dynamic = ZohoConnectorPro()

dynamic_results = auto_scan_and_respond_dynamic(live_connector_dynamic, buffer_threshold_days=10.0)
display(HTML('<h3 style="margin:18px 0 6px 0;">Scan Summary -- All Live Items</h3>'))
if not dynamic_results.empty:
    display(style_scan_results(dynamic_results))
else:
    print('No items to summarize.')

vendor_scorecard = analyze_vendors(live_connector_dynamic)
render_vendor_scorecard(vendor_scorecard)


## 15. Fully automatic — continuous monitoring loop

`run_forever()` re-runs `auto_scan_and_respond()` on a fixed interval for as long as this cell keeps executing — you start it once and it keeps checking live Zoho data on its own, with no further manual calls. A fresh connector is created each cycle so its OAuth token and caches don't go stale.

**Being honest about what 'fully automatic' means here:** this runs automatically *while this Colab cell is executing* — that's genuinely hands-off monitoring, not a one-shot script. But a free Colab session disconnects after a period of idle time and has a hard runtime ceiling (commonly cited around 12 hours), and closing the browser tab or losing your internet connection stops it. That's a real constraint of the platform, not something this code can work around. For monitoring that survives your laptop being closed — true 24/7 unattended automation — the same `ZohoConnectorPro`/agent code would need to run somewhere that stays on regardless of your session: Colab Enterprise's built-in notebook scheduling (a paid Google Cloud product, different from the free Colab you're using), a small cron job on a server, or a scheduled cloud function. That's a deployment change, not a code change — the agents and connector above would move over unchanged.

In [ ]:
def run_forever(connector_factory, buffer_threshold_days=10.0, interval_minutes=60, max_cycles=None,
                scan_fn=None):
    """Runs the automatic scan on a loop. connector_factory is a zero-arg callable returning
    a fresh connector each cycle. scan_fn defaults to auto_scan_and_respond_dynamic (Section 18) --
    the fully dynamic, no-hardcoded-part-list scan -- but you can pass the older
    auto_scan_and_respond if you still want the PART_TO_MODELS-based behavior.
    Set max_cycles for testing; leave it None to run until you interrupt the cell
    (Colab: the stop/square button) or the session ends."""
    if scan_fn is None:
        scan_fn = auto_scan_and_respond_dynamic
    cycle = 0
    all_summaries = []
    while max_cycles is None or cycle < max_cycles:
        cycle += 1
        print(f"\n{'#'*72}\n# CYCLE {cycle} -- {datetime.now().isoformat(timespec='seconds')} UTC\n{'#'*72}")
        try:
            connector = connector_factory()
            results = scan_fn(connector, buffer_threshold_days=buffer_threshold_days)
            results['cycle'] = cycle
            results['timestamp_utc'] = datetime.now().isoformat(timespec='seconds')
            all_summaries.append(results)
        except Exception as e:
            print(f'[run_forever] cycle {cycle} failed, will retry next cycle rather than stopping: {e}')
        if max_cycles is None or cycle < max_cycles:
            print(f'\nSleeping {interval_minutes} min until next automatic scan (interrupt the cell to stop)...')
            time.sleep(interval_minutes * 60)
    print('\nMonitoring loop ended.')
    return pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()

print('run_forever() defined. Example: run_forever(lambda: ZohoConnectorPro(), interval_minutes=60)')

## 16. Start it

This is the one cell you actually run to turn everything on. Adjust `interval_minutes` and `buffer_threshold_days` to your needs. Leave `max_cycles=None` to run indefinitely (until you stop the cell); it's set to a small number here so a first run doesn't sit forever by accident.

In [ ]:

history = run_forever(
    connector_factory=lambda: ZohoConnectorPro(),
    buffer_threshold_days=10.0,
    interval_minutes=1,
    max_cycles=2,
)

display(HTML('<h3 style="margin:18px 0 6px 0;">Monitoring History — All Cycles</h3>'))
if not history.empty:
    display(style_scan_results(history))
else:
    print('No cycles completed.')


## ⚠ Security note -- credentials were hardcoded in plaintext below

The two cells that follow originally had a real `ZOHO_CLIENT_SECRET`, `ZOHO_REFRESH_TOKEN`, and `ZOHO_ORG_ID` typed directly into the notebook. That's a live credential leak the moment this notebook is shared, committed to git, or opened by anyone else -- a refresh token alone is enough for someone to pull your inventory and vendor data or create purchase orders in your Zoho org.

**If those values were real, treat them as compromised now:** revoke that Zoho OAuth client (or regenerate its client secret) and generate a fresh refresh token from [Zoho API Console](https://api-console.zoho.in/) before relying on this notebook again. The cells below have been rewritten to read from Colab Secrets instead -- the same pattern `ZohoConnector` already uses in Section 9 -- so no secret value ever needs to sit in the notebook text itself. Add `ZOHO_CLIENT_ID`, `ZOHO_CLIENT_SECRET`, `ZOHO_ORG_ID`, and (after the one-time exchange below) `ZOHO_REFRESH_TOKEN` under the key icon in the left Colab sidebar.


In [ ]:
# One-time step: exchange an authorization code for a refresh token.
# Run this once after generating a fresh AUTHORIZATION_CODE in the Zoho API Console,
# then copy the refresh_token from the printed response into Colab Secrets as
# ZOHO_REFRESH_TOKEN. Do NOT leave real values typed into this cell -- clear the
# AUTHORIZATION_CODE field again once you've copied the refresh token out.
from google.colab import userdata
import requests

CLIENT_ID = userdata.get('ZOHO_CLIENT_ID')
CLIENT_SECRET = userdata.get('ZOHO_CLIENT_SECRET')
AUTHORIZATION_CODE = ''  # paste a freshly generated code here only for this one run, then clear it

if not AUTHORIZATION_CODE:
    print('Set AUTHORIZATION_CODE above (temporarily) to run the one-time token exchange.')
else:
    response = requests.post('https://accounts.zoho.in/oauth/v2/token', params={
        'code': AUTHORIZATION_CODE,
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
        'grant_type': 'authorization_code',
    })
    print(response.json())
    print('\nCopy the refresh_token above into Colab Secrets as ZOHO_REFRESH_TOKEN, '
          'then clear AUTHORIZATION_CODE from this cell.')


In [ ]:
# Sanity-check that Colab Secrets are wired up correctly (no real values printed).
from google.colab import userdata
import requests

token_response = requests.post('https://accounts.zoho.in/oauth/v2/token', params={
    'refresh_token': userdata.get('ZOHO_REFRESH_TOKEN'),
    'client_id': userdata.get('ZOHO_CLIENT_ID'),
    'client_secret': userdata.get('ZOHO_CLIENT_SECRET'),
    'grant_type': 'refresh_token',
})
token_data = token_response.json()
print('Access token obtained.' if 'access_token' in token_data else token_data)
